# A Modern Solution to a Modern Problem

## Find jobs quick and enter the market quicker.

### CHALLENGE:

It is hard to keep track of the job market and find the right jobs to apply for.

You can now find the right jobs that match your resume in few minutes across domains and eliminate duplicate applications. 

Simplify the process and get more done!

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from pypdf import PdfReader

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4.1'
openai = OpenAI()

API key looks good so far


## First step: Fetch the domain and gather all the job recent job applications in your domain. 

### Use the Beautiful Soup library to scrape your favorite job board and gather all the recent posts. 

It will gather all the links posted in your domain that is aimed for juniors and associates posted within one day of posting.

In [3]:
link_system_prompt = """
You are provided with a list of links found on a job board webpage.
You are able to decide which of the links would be most relevant to a job posting,
such as links to specific job postings, or links to the company's website.
Ensure that the job description is a summary of the job duties and not the entire job description.
You should respond in JSON as in this example:

{
    "links": [
        {"Company name": "name", "Job title": "title", "Job duties": "summary of the job duties", "url": "https://full.url/goes/here/about"},
        {"Company name 2": "name 2", "Job title 2": "title", "Job duties 2": "summary of the job duties 2", "url": "https://another.full.url/careers"}
    ]
}

- Always respond in JSON format.
- Always include company name, job title, job duties, and url.
"""

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a job posting, not the company's website.
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links, or links to the company's website.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [5]:
print(get_links_user_prompt("https://www.linkedin.com/jobs/search/?f_E=2%2C3&f_TPR=r86400&geoId=104769905&keywords=""ai""&origin=JOB_SEARCH_PAGE_JOB_FILTER&refresh=true"))


Here is the list of links on the website https://www.linkedin.com/jobs/search/?f_E=2%2C3&f_TPR=r86400&geoId=104769905&keywords=ai&origin=JOB_SEARCH_PAGE_JOB_FILTER&refresh=true -
Please decide which of these are relevant web links for a job posting, not the company's website.
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links, or links to the company's website.

Links (some might be relative links):

#main-content
/?trk=public_jobs_nav-header-logo
https://www.linkedin.com/login?emailAddress=&fromSignIn=&fromSignIn=true&session_redirect=https%3A%2F%2Fwww.linkedin.com%2Fjobs%2Fsearch%2F%3Ff_E%3D2%252C3%26f_TPR%3Dr86400%26geoId%3D104769905%26keywords%3Dai%26origin%3DJOB_SEARCH_PAGE_JOB_FILTER%26refresh%3Dtrue&trk=public_jobs_nav-header-signin
https://www.linkedin.com/signup/cold-join?source=jobs_registration&session_redirect=https%3A%2F%2Fwww.linkedin.com%2Fjobs%2Fsearch%2F%3Ff_E%3D2%252C3%26f_TPR%3Dr86400%26geoId%3D104769905%26keywords%

In [16]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [41]:
jobs = select_relevant_links("https://www.linkedin.com/jobs/search/?f_E=2%2C3&f_TPR=r86400&geoId=104769905&keywords=""ai""&origin=JOB_SEARCH_PAGE_JOB_FILTER&refresh=true")

Selecting relevant links for https://www.linkedin.com/jobs/search/?f_E=2%2C3&f_TPR=r86400&geoId=104769905&keywords=ai&origin=JOB_SEARCH_PAGE_JOB_FILTER&refresh=true by calling gpt-4.1
Found 15 relevant links


## Second step: match with personal experience!

# Filter the jobs to match your experience!

Update your resume.pdf and the LLM will make the decision for you and reason why you should apply. 

In [31]:
name = "Sriram Karthikeyan"

In [59]:
reader = PdfReader("Personal_details/resume.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [43]:
print(resume)

Sriram Karthikeyan
sriramkofc@gmail.com
 
+61 426354114
 
Sydney, NSW
 
https://www.linkedin.com/in/sriramkarthikeyan/
 
PROFILE
Working on production AI systems used within financial advisory workflows, with a focus on reliability, 
compliance, and practical business value. Designed and implemented RAG pipelines to analyse advice 
transcripts and supporting documents, enabling context-aware reasoning and structured insight extraction. 
Built agent-style automation logic to assess intent, tone, and risk indicators, supporting internal compliance 
and quality review processes. Developed end-toend automated workflows covering file ingestion, analysis, 
validation, and reporting, reducing manual effort and improving turnaround times. Collaborate closely with 
product, compliance, and leadership stakeholders to ensure AI outputs align with regulatory and 
operational requirements.
EDUCATION
08/2022 – 07/2024
Sydney, NSW
Masters in IT and IT Management, The University of Sydney
Data Managem

In [55]:
suggestion_system_prompt = f"""You are a helpful assistant helping {name} filter jobs that are best to apply for.

You act as a career counselor, reviewing job postings and recommending the most relevant opportunities based on {name}'s resume, background, skills, and experience.

Go through the provided job postings and select only the ones worth applying for. For each recommended job, explain your reasoning clearly.

If you are unsure whether a job is a good fit, include it with a note in the "Reason" field explaining your uncertainty. Only return jobs worth considering — omit poor matches entirely.
"""

suggestion_system_prompt += f"\n\n## Resume:\n{resume}\n\n"
suggestion_system_prompt += f"Use this context to make decisions on the jobs {name} should apply to. Always include company name, job title, job duties, reason and url."

In [39]:
def get_suggestion_user_prompt(jobs):
    user_prompt = f"""
You are looking at a list of jobs: {jobs}
Use this information to build a shorter list of jobs that are most relevant to {name}'s career, background, skills and experience.\n\n
"""
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [44]:
get_suggestion_user_prompt(jobs)

"\nYou are looking at a list of jobs: {'links': [{'Company name': 'YO IT Consulting', 'Job title': 'AI Trainer / Data Annotator (Remote)', 'Job duties': 'Label and annotate data for machine learning models, participate in data quality audits, and provide feedback on AI system performance.', 'url': 'https://au.linkedin.com/jobs/view/ai-trainer-data-annotator-remote-at-yo-it-consulting-4403689784?position=1&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=dUcbOItupNnxQHi04KMjjg%3D%3D'}, {'Company name': 'Apple', 'Job title': 'Generative AI Software Engineer', 'Job duties': 'Design and develop generative AI models for consumer-facing products, collaborate with cross-functional teams, and optimize software performance.', 'url': 'https://au.linkedin.com/jobs/view/generative-ai-software-engineer-at-apple-4403933931?position=2&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=e3uJbkn9MbuOgNAPijVyWw%3D%3D'}, {'Company name': 'Rassure', 'Job title': 'Junior Data Engineer / Analyst'

In [57]:
def create_job_list():
    response = openai.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": suggestion_system_prompt},
            {"role": "user", "content": get_suggestion_user_prompt(jobs)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [58]:
create_job_list()

Here are the jobs Sriram Karthikeyan should consider applying to, based on his background in AI systems, automation, data analysis, and practical business applications within regulated industries:

---

### 1. Apple – Generative AI Software Engineer

- **Job duties:** Design and develop generative AI models for consumer-facing products, collaborate with cross-functional teams, and optimize software performance.
- **Reason:** Sriram’s strong foundation in production AI systems, prompt engineering, and pipeline development aligns closely with the responsibilities here. His experience with regulatory, reliable, and scalable AI solutions is valuable at Apple, where product robustness and cross-team collaboration are crucial.
- **URL:** [Apple – Generative AI Software Engineer](https://au.linkedin.com/jobs/view/generative-ai-software-engineer-at-apple-4403933931?position=2&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=e3uJbkn9MbuOgNAPijVyWw%3D%3D)

---

### 2. Alignerr – Machine Learning Engineer

- **Job duties:** Develop and implement machine learning models, analyze large datasets, fine-tune algorithms, and collaborate with team members on AI projects.
- **Reason:** This role is an excellent technical match, leveraging Sriram’s experience building RAG pipelines, agent-style automation, and advanced ML workflows. Alignerr’s focus on model development and data analysis aligns well with his technical toolkit (Python, SQL, Azure, FastAPI).
- **URL:** [Alignerr – Machine Learning Engineer](https://au.linkedin.com/jobs/view/machine-learning-engineer-at-alignerr-4404563895?position=4&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=eMbuNEzrRMcbIhfQueRlew%3D%3D)

---

### 3. Rassure – Junior Data Engineer / Analyst

- **Job duties:** Support data engineering tasks, analyze datasets to derive actionable insights, and assist in optimizing data workflows.
- **Reason:** This role matches Sriram’s previous data background (at UnfoldLabs and as a Software Developer Intern) and technical skills (Python, SQL, MongoDB). It’s an early-career role, so it may be a fallback position, but is highly relevant and would solidify his data engineering experience.
- **URL:** [Rassure – Junior Data Engineer / Analyst](https://au.linkedin.com/jobs/view/junior-data-engineer-analyst-at-rassure-4403909274?position=3&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=IIQYJYGvs4R9EwtOXsrTkA%3D%3D)

---

### 4. Westpac Institutional and Business – Junior Quantitative Engineer - CTT

- **Job duties:** Support quantitative analytics, develop and validate models, and contribute to financial product development through quantitative research.
- **Reason:** Sriram’s experience in the intersection of AI and finance (FinTalkr), plus data-driven approaches, aligns well. This could leverage his technical AI foundation while adding quantitative finance experience, especially pertinent in regulated, compliance-heavy environments.
- **URL:** [Westpac – Junior Quantitative Engineer – CTT](https://au.linkedin.com/jobs/view/junior-quantitative-engineer-ctt-at-westpac-institutional-and-business-4404556109?position=8&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=lrzFKhadc3lL2n71hZYOIg%3D%3D)

---

### 5. Humanforce – Commercial Data Analyst

- **Job duties:** Analyze commercial data, generate business intelligence reports, and provide recommendations to improve strategies and performance.
- **Reason:** Sriram’s reporting and analytics work (automating insights, creating dashboards) fits well. This role is less AI-specific but builds on his data analysis and business reporting experience for broader commercial exposure.
- **URL:** [Humanforce – Commercial Data Analyst](https://au.linkedin.com/jobs/view/commercial-data-analyst-at-humanforce-4401903509?position=5&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=Dgybkk%2B8PI2%2BPOTwKkjmzg%3D%3D)

---

### 6. Livewire Markets – Data Analyst

- **Job duties:** Conduct data analysis, generate actionable business insights, and support strategy development through data-driven recommendations.
- **Reason:** Sriram’s skills in Python, SQL, and analytics, plus financial domain exposure, make this a good match for a data-driven role supporting decision-making in a finance-adjacent environment.
- **URL:** [Livewire Markets – Data Analyst](https://au.linkedin.com/jobs/view/data-analyst-at-livewire-markets-4404837072?position=7&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=qwnok5pXjWZlihnqija0zw%3D%3D)

---

### 7. IBM – Graduate Associate Experience Consultant – Customer Experience Transformation

- **Job duties:** Assist in customer experience transformation projects, analyze client needs, and help implement digital solutions to improve client experiences.
- **Reason:** This is at the intersection of digital transformation (where Sriram has experience with automation and workflow design) and client-facing analytics. If considering consulting/strategy as a path, this could be a good entry point leveraging technology and business understanding.
- **URL:** [IBM – Graduate Associate Experience Consultant](https://au.linkedin.com/jobs/view/graduate-associate-experience-consultant-customer-experience-transformation-at-ibm-4381726427?position=6&pageNum=0&refId=xZL0yMEZp4Y0BhYnzJhiqA%3D%3D&trackingId=pQWlDBm%2BbNdWa8tkw7SPPg%3D%3D)

---

#### Discussion of Excluded Roles:
- **YO IT Consulting — AI Trainer/Data Annotator:** This role is low-skill compared to Sriram’s capabilities and experience in developing, not just labeling, AI solutions.
- **Smartways Logistics — Operational Insights Analyst:** While analytical, it is tightly focused on logistics, where Sriram’s domain experience is limited.
- **Everis — Social Media AI Video Content Creator:** This is a primarily creative content role, not aligned with his technical/data focus.
- **Marloo — GTM Associate:** Go-to-market and product launch roles are more business/marketing than technical/AI focused.
- **The Missing Link — Project Coordinator - Cyber Security PMO:** Project coordination in cybersecurity is a significant pivot from AI/data engineering/development.

---

If you would like to narrow the list by preferred domain (AI/ML development vs. analytics vs. consulting), I can further prioritize these options.